In [ ]:
!pip install "unsloth[torch]" trl datasets crewai transformers accelerate


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.1 MB/s eta 0:00:00
INFO: pip is still looking at multiple versions of transformers to determine which version is compatible with other requirements. T

In [ ]:
!pip install LiteLLM

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 122.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 28.5 MB/s eta 0:00:00


In [ ]:
!pip install groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 5.8 MB/s eta 0:00:00


In [ ]:
import torch
from unsloth import FastLanguageModel
from groq import Groq

bio_path = "/content/drive/MyDrive/DAPT+SFT/tinyllama-dpo-merged-final"

bio_model, bio_tok = FastLanguageModel.from_pretrained(
    model_name=bio_path,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

if bio_tok.pad_token_id is None:
    bio_tok.pad_token = bio_tok.eos_token

def make_prompt(instruction: str) -> str:
    return f"### Instruction:\n{instruction}\n\n### Response:\n"

gen_kwargs = dict(
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    repetition_penalty=1.2,
    top_p=0.9,
    eos_token_id=bio_tok.eos_token_id,
    pad_token_id=bio_tok.pad_token_id,
)

def generate_answer_local(model, tok, instruction):
    """Run inference on BioLLama-R locally."""
    prompt = make_prompt(instruction)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)
    text = tok.decode(out[0], skip_special_tokens=True)
    return text.split("### Response:\n")[-1].strip()


==((====))==  Unsloth 2026.1.3: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:

client = Groq(api_key="")


def generate_answer_groq(instruction: str, model="openai/gpt-oss-120b"):
    """Run inference using Groq-hosted LLM."""
    completion = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": instruction}],
        temperature=0.7,
        max_completion_tokens=512,
        top_p=0.9,
        reasoning_effort="medium",
        stream=False,
    )
    return completion.choices[0].message.content.strip()

In [ ]:

def clinical_intake_agent(user_query: str):
    """Groq decides if query is medical or non-medical."""
    instruction = f"Classify the following query as 'medical' or 'non-medical':\n\n{user_query}"
    category = generate_answer_groq(instruction)
    return {"normalized_query": user_query.strip(), "category": category.lower()}

def medical_reasoning_agent(normalized_query: str):
    """BioLLama-R does medical reasoning locally."""
    return generate_answer_local(bio_model, bio_tok, normalized_query)

def safety_agent(response: str):
    """Groq validates and sanitizes outputs for safety/compliance."""
    instruction = f"""Review the following response and ensure it is safe:
- No definitive diagnoses
- No prescriptions or dosages
- No unsafe medical instructions
If unsafe, rewrite safely. If safe, return as is.

Response:
{response}
"""
    return generate_answer_groq(instruction)

def agentic_pipeline(user_query: str):
    print("=" * 80)
    print(f"User Query: {user_query}\n")

    intake = clinical_intake_agent(user_query)
    print(f"[Intake Agent] Category: {intake['category']}\n")

    if intake["category"] == "medical":
        reasoning = medical_reasoning_agent(intake["normalized_query"])
        print(f"[Reasoning Agent] Raw output:\n{reasoning}\n")

        safe_output = safety_agent(reasoning)
        print(f"[Safety Agent] Final output:\n{safe_output}\n")
    else:
        print("[Pipeline] Non-medical query. No medical reasoning performed.\n")

questions = [
    "Patient reports chest pain and shortness of breath. What should be done?",
]

for q in questions:
    agentic_pipeline(q)

User Query: Patient reports chest pain and shortness of breath. What should be done?

[Intake Agent] Category: medical

[Reasoning Agent] Raw output:
Chest pain and shortness of breath indicate acute coronary syndrome (ACS). The patient is recommended to be evaluated immediately by the emergency department or hospitalized at a local medical center. It's important for the physician to rule out any other conditions that may cause similar symptoms, such as heart failure or hypertension. Emergent cardiac arrest is not required in most cases but requires immediate treatment with appropriate therapy. Once stabilized, the patient needs further evaluation and management within the next 24 hours. Cardiopulmonary resuscitation can prolong survival times. Medical intervention can improve functional capacity after recovery from the initial shock and help prevent long-term complications like chronic heart disease. Patients requiring invasive procedures require careful monitoring during and followin